In [48]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Pharmacovigilance/ADR.csv')

In [50]:
df

,text,drug,effect,indexes
0,Intravenous azithromycin-induced ototoxicity.,azithromycin,ototoxicity,"{'drug': {'start_char': [12], 'end_char': [24]..."
1,"Immobilization, while Paget's bone disease was...",dihydrotachysterol,increased calcium-release,"{'drug': {'start_char': [91], 'end_char': [109..."
2,Unaccountable severe hypercalcemia in a patien...,dihydrotachysterol,hypercalcemia,"{'drug': {'start_char': [84], 'end_char': [102..."
3,METHODS: We report two cases of pseudoporphyri...,naproxen,pseudoporphyria,"{'drug': {'start_char': [58], 'end_char': [66]..."
4,METHODS: We report two cases of pseudoporphyri...,oxaprozin,pseudoporphyria,"{'drug': {'start_char': [71], 'end_char': [80]..."
...,...,...,...,...
6816,Lithium treatment was terminated in 1975 becau...,Lithium,lithium intoxication,"{'drug': {'start_char': [0], 'end_char': [7]},..."
6817,Lithium treatment was terminated in 1975 becau...,lithium,lithium intoxication,"{'drug': {'start_char': [52], 'end_char': [59]..."
6818,Eosinophilia caused by clozapine was observed ...,clozapine,Eosinophilia,"{'drug': {'start_char': [23], 'end_char': [32]..."
6819,Eosinophilia has been encountered from 0.2 to ...,clozapine,Eosinophilia,"{'drug': {'start_char': [55], 'end_char': [64]..."


In [51]:
df.shape

(6821, 4)

In [52]:
print(df['text'].duplicated().sum())

2550


In [53]:
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(len(df))

4271


In [54]:
df

,text,drug,effect,indexes
0,Intravenous azithromycin-induced ototoxicity.,azithromycin,ototoxicity,"{'drug': {'start_char': [12], 'end_char': [24]..."
1,"Immobilization, while Paget's bone disease was...",dihydrotachysterol,increased calcium-release,"{'drug': {'start_char': [91], 'end_char': [109..."
2,Unaccountable severe hypercalcemia in a patien...,dihydrotachysterol,hypercalcemia,"{'drug': {'start_char': [84], 'end_char': [102..."
3,METHODS: We report two cases of pseudoporphyri...,naproxen,pseudoporphyria,"{'drug': {'start_char': [58], 'end_char': [66]..."
4,"Naproxen, the most common offender, has been a...",Naproxen,erythropoietic protoporphyria,"{'drug': {'start_char': [0], 'end_char': [8]},..."
...,...,...,...,...
4266,Reversible nonthrombocytopenic palpable purpur...,metoclopramide,nonthrombocytopenic palpable purpura,"{'drug': {'start_char': [64], 'end_char': [78]..."
4267,Lithium treatment was terminated in 1975 becau...,Lithium,diabetes insipidus-like syndrome,"{'drug': {'start_char': [0], 'end_char': [7]},..."
4268,Eosinophilia caused by clozapine was observed ...,clozapine,Eosinophilia,"{'drug': {'start_char': [23], 'end_char': [32]..."
4269,Eosinophilia has been encountered from 0.2 to ...,clozapine,Eosinophilia,"{'drug': {'start_char': [55], 'end_char': [64]..."


In [55]:
import ast

In [56]:
sentence_spans = {}

for text, indexes in zip(df["text"], df["indexes"]):

    if isinstance(indexes, str):
        indexes = ast.literal_eval(indexes)

    if text not in sentence_spans:
        sentence_spans[text] = []

    for entity_type, key in [("DRUG", "drug"), ("EFFECT", "effect")]:
        positions = indexes[key]

        for start, end in zip(positions["start_char"], positions["end_char"]):
            span = {"start": start, "end": end, "label": entity_type}

            if span not in sentence_spans[text]:
                sentence_spans[text].append(span)

ner_df = pd.DataFrame({
    "text": list(sentence_spans.keys()),
    "spans": list(sentence_spans.values())
})

print(ner_df.shape)

(4271, 2)


In [57]:
from sklearn.model_selection import train_test_split
train, temp = train_test_split(df, test_size=0.2 , random_state = 42)
val, test = train_test_split(temp, test_size = 0.5 , random_state = 42)

In [58]:
train.shape , val.shape , test.shape

((3416, 4), (427, 4), (428, 4))

In [59]:

from datasets import Dataset
train_dataset = Dataset.from_pandas(train.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test.reset_index(drop=True))

